<a href="https://colab.research.google.com/github/AgacheM/Analyzing-Toronto-Airbnb-Dataset/blob/main/Notebook%203%20-%20Manual%20Exploratory%20Data%20Analysis%20of%20Reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Import Libraries**

In [1]:
#Check the version of Python being run
!python --version

# Install all dependencies
!pip install pandas matplotlib seaborn ipython numpy

#Import Libraries
#Data Manipulation, Cleaning, and Analysis
import pandas as pd
import numpy as np

#Formatting and Visualizations
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from IPython.display import FileLink
from IPython.display import IFrame

#Format without scientific notation
pd.options.display.float_format = '{:,.0f}'.format

Python 3.12.13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.0 MB/s eta 0:00:00


# **2. Download Reviews Data from Insideairbnb.com**

In [2]:
#Import Reviews files
url_toronto = "https://data.insideairbnb.com/canada/on/toronto/2025-11-11/data/reviews.csv.gz"
df_toronto = pd.read_csv(url_toronto, compression='gzip')

url_montreal = "https://data.insideairbnb.com/canada/qc/montreal/2025-09-18/data/reviews.csv.gz"
df_montreal = pd.read_csv(url_montreal, compression='gzip')

url_ottawa = "https://data.insideairbnb.com/canada/on/ottawa/2025-09-22/data/reviews.csv.gz"
df_ottawa = pd.read_csv(url_ottawa, compression='gzip')

#Add city labels
df_toronto['city'] = 'Toronto'
df_montreal['city'] = 'Montreal'
df_ottawa['city'] = 'Ottawa'

#Combine data
df_listings = pd.concat([df_toronto, df_montreal, df_ottawa], ignore_index=True)

#Encode city labels for future modelling
df_listings['city_code'] = df_listings['city'].map({
    'Toronto': 1,
    'Montreal': 2,
    'Ottawa': 3})

# **3. Check Common Features in Combined Datasets**

In [3]:
#Compare number of columns to ensure they are the same between cities
print("Toronto columns:", len(df_toronto.columns))
print("Montreal columns:", len(df_montreal.columns))
print("Ottawa columns:", len(df_ottawa.columns))

#Keep only the common columns, use a set to remove duplicates and ignore order of columns
common_cols = list(
    set(df_toronto.columns) &
    set(df_montreal.columns) &
    set(df_ottawa.columns))

print(f"""
Check all 3 cities' datasets contain the same column names = {set(df_toronto.columns) == set(df_ottawa.columns) == set(df_montreal.columns)}.
      """)

print(f"""The datsets of the 3 cities share these column columns: {common_cols}
      """)

#Check row counts
print(f"""The number of rows / observations in each city are:
{df_listings['city'].value_counts()}""")

Toronto columns: 7
Montreal columns: 7
Ottawa columns: 7

Check all 3 cities' datasets contain the same column names = True.
      
The datsets of the 3 cities share these column columns: ['city', 'date', 'listing_id', 'comments', 'reviewer_name', 'id', 'reviewer_id']
      
The number of rows / observations in each city are:
city
Toronto     651546
Montreal    403093
Ottawa      141096
Name: count, dtype: int64


# **4. Dataset Overview**

In [4]:
#Summarize Listings Data
print(f"""
DATASET OVERVIEW
----------------
Rows: {df_listings.shape[0]:,}
Columns: {df_listings.shape[1]}

FIRST 5 ROWS
---------------""")
display(df_listings.head())


DATASET OVERVIEW
----------------
Rows: 1,195,735
Columns: 8

FIRST 5 ROWS
---------------


,listing_id,id,date,reviewer_id,reviewer_name,comments,city,city_code
0,9974111,105972997,2016-10-03,74630712,Lily,I absolutely loved the location!! I was there ...,Toronto,1
1,9974111,125205050,2017-01-04,23609,Mark,"Stayed here for 2 weeks , great location , gre...",Toronto,1
2,9974111,191232367,2017-09-06,147270253,Dianne,Awesome location! Communication with Shoug was...,Toronto,1
3,9974111,196001139,2017-09-21,5590969,Charie,Had a really great two week stay at Shoug's pl...,Toronto,1
4,9974111,197975881,2017-09-27,76030814,Oswald,great and easy stay - no issues at all,Toronto,1


# **4.Basic Data Profiling of Numeric Features**
  For further analysis, please see the file [Notebook 1 - Automated Exploratory Data Analysis](https://github.com/AgacheM/Analyzing-Toronto-Airbnb-Dataset/blob/main/Notebook%201%20-%20Automated%20Exploratory%20Data%20Analysis%20-%20Data%20Types%2C%20Distributions%2C%20Skewness%2C%20and%20Correlations.ipynb)


In [11]:
#Columns, Basic Data Types, Missing values
print("OVERVIEW OF KEY NUMERICAL STATS, DATA TYPES AND MISSING VALUES \n ----------------------------------------------------------")

display(df_listings.describe())



OVERVIEW OF KEY NUMERICAL STATS, DATA TYPES AND MISSING VALUES 
 ----------------------------------------------------------


,listing_id,id,city_code
count,"1,195,549","1,195,549","1,195,549"
mean,"392,435,127,927,097,216","809,530,693,180,847,616",2
std,"493,294,934,634,440,256","554,571,630,944,654,400",1
min,"1,419","7,830",1
25%,"20,712,899","705,785,968",1
50%,"46,808,603","952,247,579,734,143,360",1
75%,"846,312,826,127,190,400","1,281,245,957,646,013,184",2
max,"1,548,112,398,843,094,272","1,561,026,480,266,877,440",3


# **5. Identify and Handle Missing Values**

In [6]:
#Calculate Missing Values
print ("Missing Values:")
missing = df_listings.isnull().sum()
missing = missing[missing>0]

missing_percent=(missing/len(df_listings))*100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent
})

#View only columns with missing values
missing_summary = missing_summary[missing_summary["Missing Count"]>0]

#Correct Errors
if len(missing_summary) > 0:
    display(
        missing_summary
        .sort_values (by="Missing %", ascending= False)
        .style.format({
            "Missing Count": "{:,.0f}",
            "Missing %": "{:.2f}%"
        })
        .background_gradient(cmap="Reds")
    )
else:
    print ("No Missing Values Found")

Missing Values:


,Missing Count,Missing %
comments,186,0.02%
reviewer_name,2,0.00%


In [8]:
#Drop rows representing the missing values under comments
print ("Drop rows with missing colunns.")

df_listings = df_listings.dropna(subset=["comments"])


print ("Remaning Missing Values:\n")
missing = df_listings.isnull().sum()
missing = missing[missing>0]

missing_percent=(missing/len(df_listings))*100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_percent
})

#View only columns with missing values
missing_summary = missing_summary[missing_summary["Missing Count"]>0]

#Correct Errors
if len(missing_summary) > 0:
    display(
        missing_summary
        .sort_values (by="Missing %", ascending= False)
        .style.format({
            "Missing Count": "{:,.0f}",
            "Missing %": "{:.2f}%"
        })
        .background_gradient(cmap="Reds")
    )
else:
    print ("No Missing Values Found")

Drop rows with missing colunns.
Remaning Missing Values:



,Missing Count,Missing %
reviewer_name,2,0.00%


# **7. Manual Feature Selection: Drop Columns Containing Personally Identifiable Information (PII) or not Adding Analytical Value**

In [10]:
#The columns below will be dropped, due to  PII concerns, low variablity, or in the case of metadata, due to a lack of signifiant value in determining price

more_columns_to_drop = [
#Reviewer Info or PII
'reviewer_name',
#Metadata
'date','reviewer_id']

df_listings = df_listings.drop(columns=more_columns_to_drop, errors='ignore')

print (f"""Dropped {len(more_columns_to_drop)} columns representing PII or not adding analytical value.""")

Dropped 3 columns representing PII or not adding analytical value.
